# Aula 1 — Python no contexto de dados: por onde começar

> **Objetivo:** entender onde Python se encaixa no stack de dados e escrever os primeiros scripts úteis.

---

## Por que Python?

Você já conhece Data Lakes, LakeHouse, Data Mesh. Mas quem **escreve** os dados nessas camadas? Quem faz a ingestão, a transformação, a validação?

Na maioria das empresas: **Python**.

- Airflow escreve DAGs em Python
- Spark tem uma API Python (PySpark)
- dbt suporta macros e scripts Python
- Glue, Lambda, Dataflow — todos aceitam Python

SQL descreve *o que* você quer. Python descreve *como* chegar lá.

## 1. O ambiente

Você pode rodar Python de três formas:

| Forma | Quando usar |
|---|---|
| **Jupyter / Colab** | Exploração, aprendizado, análise |
| **Script `.py`** | Produção, agendamento, pipelines |
| **Terminal (`python`)** | Testes rápidos |

Neste curso usamos **notebooks** — cada célula é um bloco de código que você executa individualmente.

> 💡 Para executar uma célula: **Shift + Enter**

In [1]:
# Vamos começar calculando o tamanho de um arquivo em MB
tamanho_bytes = 524_288_000  # 500 MB em bytes
tamanho_mb = tamanho_bytes / 1_048_576

print('Tamanho do arquivo:', tamanho_mb, 'MB')

Tamanho do arquivo: 500.0 MB


**📝 O que essa célula faz:**

- `tamanho_bytes = 524_288_000` → cria uma variável chamada `tamanho_bytes` e guarda o número 524.288.000 nela.
- `tamanho_mb = tamanho_bytes / 1_048_576` → divide o valor em bytes por 1.048.576 (que é `1024 * 1024`, ou seja, **1 MB em bytes**). O resultado é o tamanho em megabytes.
- `print(...)` → imprime o resultado na tela, juntando texto e variável.

**🔢 Por que o `_` dentro do número?**

`524_288_000` é exatamente o mesmo número que `524288000` — o `_` (underscore) é apenas um **separador visual** que o Python ignora ao calcular. Ele existe só para facilitar a leitura humana, igual usamos pontos em `500.000` no dia a dia.

```python
524_288_000 == 524288000   # True — são o mesmo número
```

Sem o `_`, contar os zeros de `524288000` é difícil e fácil de errar. Com o `_`, fica claro que são **524 milhões, 288 mil**. Isso é uma boa prática em código de dados, onde números grandes (bytes, registros, IDs) aparecem o tempo todo.

**💡 Por que dividir por 1.048.576 e não por 1.000.000?**

Porque memória e armazenamento em computação são medidos em potências de 2, não de 10. `1 MB = 1024 KB` e `1 KB = 1024 bytes`, então `1 MB = 1024 × 1024 = 1.048.576 bytes`. Se você dividisse por 1 milhão, o resultado ficaria levemente errado.

## 2. Tipos de dados que aparecem em pipelines

Quando você lê um registro de um Data Lake, encontra alguns tipos básicos:

In [3]:
# Texto (string) — nomes, IDs, categorias, datas como texto
nome_tabela = 'bronze.eventos_clientes'
status = 'processado'

# Números inteiros — contagens, IDs numéricos
quantidade_registros = 1_250_000

# Números decimais — valores monetários, métricas
taxa_erro = 0.0023

# Verdadeiro ou Falso (boolean) — flags, condições
particionado = True
vazio = False

# Nada (None) — campo ausente, nulo, sem valor
data_exclusao = None

print('Tabela:', nome_tabela)
print('Registros:', quantidade_registros)
print('Taxa de erro:', taxa_erro)
print('Particionado:', particionado)
print('Data exclusão:', data_exclusao)

Tabela: bronze.eventos_clientes
Registros: 1250000
Taxa de erro: 0.0023
Particionado: True
Data exclusão: None


**📝 O que essa célula faz:**

Cria seis variáveis, cada uma guardando um tipo diferente de dado, e imprime todas:

| Variável | Valor | Tipo |
|---|---|---|
| `nome_tabela` | `'bronze.eventos_clientes'` | texto (`str`) |
| `status` | `'processado'` | texto (`str`) |
| `quantidade_registros` | `1_250_000` | número inteiro (`int`) |
| `taxa_erro` | `0.0023` | número decimal (`float`) |
| `particionado` | `True` | verdadeiro/falso (`bool`) |
| `data_exclusao` | `None` | vazio/nulo (`NoneType`) |

**🔤 Por que aspas simples `'texto'` e não aspas duplas `"texto"`?**

Em Python, os dois funcionam exatamente igual — é só estilo. A maioria dos programadores Python prefere aspas simples por padrão, e reserva as duplas para quando o próprio texto já contém um apóstrofo (ex: `"não vou usar 'aspas' aqui"`).

**🔢 De novo o `_` em `1_250_000`**

Mesma lógica da célula anterior: `1_250_000` é só `1250000` escrito de forma mais legível — um milhão, duzentos e cinquenta mil registros.

**❓ Por que `None` e não `False`, `0` ou `""` (texto vazio)?**

Porque `None` representa **ausência de valor**, algo diferente de "falso" ou "zero". Se um cliente nunca foi excluído, o campo `data_exclusao` não deveria ser `0` (zero é uma data válida em alguns formatos!) — deveria literalmente **não ter valor nenhum**. É assim que bancos de dados representam campos nulos, e o Python usa `None` para a mesma ideia.

## 3. Verificando tipos

Sempre que receber dados de uma fonte externa, você vai querer saber o tipo do que chegou:

In [5]:
# A função type() diz o tipo de qualquer valor
print(type('bronze.eventos_clientes'))   # str
print(type(1_250_000))                   # int
print(type(0.0023))                      # float
print(type(True))                        # bool
print(type(None))                        # NoneType
isinstance(1, int)                       #Valida

<class 'str'>
<class 'int'>
<class 'float'>
<class 'bool'>
<class 'NoneType'>


True

**📝 O que essa célula faz:**

Usa a função `type()` para perguntar ao Python "qual é o tipo desse valor?" — e imprime a resposta para cada uma das variáveis criadas antes.

```python
type('bronze.eventos_clientes')   # <class 'str'>      → é texto
type(1_250_000)                   # <class 'int'>      → é número inteiro
type(0.0023)                      # <class 'float'>    → é número decimal
type(True)                        # <class 'bool'>     → é verdadeiro/falso
type(None)                        # <class 'NoneType'> → é vazio
```

**🧪 A última linha: `isinstance(1, int)`**

`isinstance(valor, tipo)` é outra forma de checar tipo, mas em vez de te dizer qual é o tipo, ela responde **sim ou não** (`True`/`False`) para a pergunta "esse valor é desse tipo?". Aqui perguntamos: "o número `1` é do tipo `int`?" — e a resposta é `True`.

**Por que isso importa em dados?** Quando você recebe um arquivo de uma API ou de outro sistema, nem sempre sabe que tipo de dado realmente chegou — um campo "número" pode ter vindo como texto (`'1500'` em vez de `1500`). Checar o tipo é o primeiro passo antes de processar.

## 4. Operações básicas úteis

As operações mais comuns em scripts de dados:

In [6]:
# Aritmética
total_bytes = 10 * 1024 * 1024 * 1024  # 10 GB
processados = 7_500_000_000
restante = total_bytes - processados
percentual = (processados / total_bytes) * 100

print(f'Processado: {percentual:.1f}%')

Processado: 69.8%


**📝 O que essa célula faz:**

Faz uma conta simples de porcentagem usando operações aritméticas:

```python
total_bytes = 10 * 1024 * 1024 * 1024   # 10 GB em bytes
processados = 7_500_000_000              # quantidade já processada
restante = total_bytes - processados      # o que falta
percentual = (processados / total_bytes) * 100   # % já feito
```

**🔢 Por que `10 * 1024 * 1024 * 1024` em vez de escrever o número direto?**

Porque assim o código **explica a si mesmo**: fica claro que é "10 GB", já que 1 GB = 1024 MB = 1024×1024 KB = 1024×1024×1024 bytes. Se você escrevesse só `10737418240`, ninguém entenderia de onde veio esse número sem fazer a conta de novo.

**🧮 Os operadores usados:**

- `*` → multiplicação
- `-` → subtração
- `/` → divisão (sempre devolve um número decimal em Python)
- `:.1f` dentro do f-string → formata o número decimal com **1 casa decimal** (explicado na próxima célula)

In [7]:
# Operações com texto
nome_arquivo = 'vendas_2024_01_15'
extensao = '.parquet'

caminho_completo = 's3://meu-lake/bronze/' + nome_arquivo + extensao
print(caminho_completo)

# Verificar se um texto contém algo
print('parquet' in caminho_completo)   # True
print(caminho_completo.upper())        # maiúsculas
print(caminho_completo.endswith('.parquet'))  # termina com?

s3://meu-lake/bronze/vendas_2024_01_15.parquet
True
S3://MEU-LAKE/BRONZE/VENDAS_2024_01_15.PARQUET
True


**📝 O que essa célula faz:**

Monta um caminho de arquivo (como se fosse um endereço no S3) **concatenando** (juntando) textos com o operador `+`, e depois testa alguns métodos de string.

```python
caminho_completo = 's3://meu-lake/bronze/' + nome_arquivo + extensao
```

Isso pega três textos e cola um atrás do outro, formando: `s3://meu-lake/bronze/vendas_2024_01_15.parquet`

**🔍 Os métodos usados depois:**

- `'parquet' in caminho_completo` → pergunta "a palavra 'parquet' aparece dentro desse texto?" → responde `True` ou `False`
- `caminho_completo.upper()` → devolve o texto inteiro em **MAIÚSCULAS** (não altera a variável original, só mostra uma versão nova)
- `caminho_completo.endswith('.parquet')` → pergunta "esse texto termina com `.parquet`?" → `True` ou `False`

**💡 Por que isso é útil em dados?** Antes de processar um arquivo, você frequentemente precisa checar a extensão (`endswith`) para saber se é CSV, JSON ou Parquet, ou procurar por palavras-chave dentro de um caminho (`in`) para identificar a camada ou origem do dado.

In [6]:
# f-strings: a forma mais limpa de montar textos com variáveis
data = '2024-01-15'
origem = 'api_pedidos'
camada = 'bronze'

caminho = f's3://datalake/{camada}/{origem}/dt={data}/data.parquet'
print(caminho)

s3://datalake/bronze/api_pedidos/dt=2024-01-15/data.parquet


**📝 O que essa célula faz:**

Usa uma **f-string** para montar um caminho de arquivo de forma mais limpa do que concatenar com `+`.

```python
caminho = f's3://datalake/{camada}/{origem}/dt={data}/data.parquet'
```

**🔤 O que é uma f-string?**

É um texto que começa com a letra `f` antes das aspas. Dentro dele, tudo que está entre chaves `{ }` é substituído pelo valor da variável correspondente. Então `{camada}` vira o valor que está guardado em `camada` (no caso, `'bronze'`), e assim por diante.

**Por que usar f-string em vez de `+`?**

Compare:
```python
# Com +  (mais difícil de ler, fácil de esquecer um espaço ou barra)
caminho = 's3://datalake/' + camada + '/' + origem + '/dt=' + data + '/data.parquet'

# Com f-string (muito mais legível)
caminho = f's3://datalake/{camada}/{origem}/dt={data}/data.parquet'
```

A f-string é o padrão usado hoje em praticamente todo código Python profissional para montar textos com variáveis.

## 5. None: o campo nulo

Em dados, campos nulos são frequentes. Python representa isso com `None`:

In [7]:
campo = None

# A forma correta de verificar se algo é None:
if campo is None:
    print('Campo vazio — ignorar ou preencher com valor padrão')
else:
    print('Campo tem valor:', campo)

# Nunca use == None (funciona, mas não é a forma Pythônica)
# Use sempre: is None  ou  is not None

Campo vazio — ignorar ou preencher com valor padrão


**📝 O que essa célula faz:**

Mostra a forma **correta** de verificar se uma variável é `None` (vazia/nula).

```python
if campo is None:
    print('Campo vazio...')
else:
    print('Campo tem valor:', campo)
```

**❓ Por que `is None` e não `== None`?**

Os dois até funcionam na prática, mas `is None` é a forma correta e recomendada pela própria comunidade Python (é uma regra do **PEP 8**, o guia oficial de estilo). A diferença técnica: `==` compara se os **valores** são iguais, enquanto `is` compara se é exatamente **o mesmo objeto** na memória. Como só existe um único `None` em todo o programa Python, usar `is` é mais rápido e mais seguro — evita bugs estranhos em casos raros onde `==` poderia se comportar de forma inesperada (por exemplo, com bibliotecas que sobrescrevem o comportamento do `==`).

**Regra prática: sempre use `is None` e `is not None`, nunca `== None`.**

---

## 🏋️ Exercício — Analisando um registro de log

Abaixo está um registro simulado de log de ingestão. Seu trabalho:

1. Imprimir o caminho completo do arquivo (origem + nome + extensão)
2. Calcular quantos MB foram processados
3. Verificar se o status é `'sucesso'`
4. Verificar se o campo `erro` está vazio

In [ ]:
# Registro de log de ingestão
origem = 's3://raw-zone/crm/'
nome_arquivo = 'clientes_20240115'
extensao = '.csv'
bytes_processados = 209_715_200  # bytes
status = 'sucesso'
erro = None

# --- Seu código abaixo ---

# 1. Caminho completo

# 2. MB processados

# 3. Verificar status

# 4. Verificar campo erro


---

## Resumo da aula

| Conceito | Exemplo |
|---|---|
| Texto | `'bronze.eventos'` |
| Número inteiro | `1_250_000` |
| Número decimal | `0.0023` |
| Verdadeiro/Falso | `True`, `False` |
| Nulo | `None` |
| f-string | `f's3://{camada}/{tabela}'` |
| Verificar nulo | `if campo is None` |

**Próxima aula:** quando os dados chegam em lote — listas e dicionários.